# Per-Sample Spike-In Limit-of-Detection Analysis

Deliberately NOT a disease-effect-size power simulation (see `perturb_genes_fold` docstring below) --
this characterizes detection sensitivity/FDR for a KNOWN, deterministic spike-in amount added on
top of each real sample's own observed count and noise floor, analogous to ERCC-style spike-in QC.
Evaluated under two independent HC/HC split designs: LOBO (held-out batches) and in-sample CV
(10-fold, pathologically miscalibrated HC samples excluded beforehand, see the QC step below).

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.Benchmark import db_hit_compare as dc
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.core.marginal_rqr import marginal_nb_rqr
from MixedEffectsModeling.core.shash import shash_transform_to_z
from MixedEffectsModeling.validation.lobo_engine import load_full_data

PERTURB_DIR = config.PERTURBATION_DIR
PERTURB_DIR.mkdir(exist_ok=True)

N_PERTURB_GENES = 500
LOG2FCS = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
N_BOOTSTRAP = 5
N_FOLDS = 10  # insample_cv only
Q_LEVELS = [0.05, 0.10, 0.20, 0.25]

ALL_SPLIT_METHODS = ["lobo", "insample_cv"]

In [ ]:
data = load_full_data()

_meta_rows = []
for bdir in sorted(config.LOBO_MIXED_DIR.iterdir()):
    meta_path = bdir / "meta.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    n_hc_test = int(sum(meta["test_is_hc"]))
    _meta_rows.append(dict(batch_id=meta["batch_id"], safe_dir=bdir.name, n_hc_test=n_hc_test))
lobo_batches = pd.DataFrame(_meta_rows).sort_values("n_hc_test", ascending=False).reset_index(drop=True)

BATCHES = ["Ward Z et al._Batch_1", "Moufarrej et al._Batch_2", "Moore et al._Batch_1",
          "Chen et al._Batch_2", "Roskams-Hieter B et al._Batch_2"]
print(lobo_batches[lobo_batches.batch_id.isin(BATCHES)])

In [ ]:
_cache = {}
name2row = {n: i for i, n in enumerate(data["names"])}
hc_meta_global = pd.read_csv(dc.ZDIR / "hc_meta.csv")
Z_hc_global = np.load(dc.ZDIR / "Z_hc_shash.npy")
gene_names_global = pickle.load(open(dc.ZDIR / "gene_names.pkl", "rb"))
gene_pos_global = {g: j for j, g in enumerate(gene_names_global)}
hc_global_rows = np.array([name2row[s] for s in hc_meta_global["sample"]])

# QC: exclude HC samples with a pathologically inflated frac(|Z|>1.96)
_frac_extreme_hc = (np.abs(Z_hc_global) > 1.96).mean(axis=1)
_q1, _q3 = np.percentile(_frac_extreme_hc, [25, 75])
HC_QC_FENCE = _q3 + 1.5 * (_q3 - _q1)
HC_QC_OUTLIER = _frac_extreme_hc > HC_QC_FENCE  # aligned to hc_meta_global / Z_hc_global rows
print(f"HC QC: excluding {HC_QC_OUTLIER.sum()}/{len(HC_QC_OUTLIER)} samples "
      f"with frac(|Z|>1.96) > {HC_QC_FENCE:.3f} (Tukey fence) from insample_cv")


def load_batch(batch_id):
    if batch_id in _cache:
        return _cache[batch_id]
    safe = lobo_batches.set_index("batch_id").loc[batch_id, "safe_dir"]
    bdir = config.LOBO_MIXED_DIR / safe
    meta = json.loads((bdir / "meta.json").read_text())
    gene_names = pickle.load(open(bdir / "gene_names.pkl", "rb"))
    Z_test = np.load(bdir / "Z_test_shash.npy")
    test_is_hc = np.array(meta["test_is_hc"])

    fits = pd.read_csv(bdir / "model_fits.csv").set_index("gene")
    fits = fits[fits["ok"]]
    shash_p = pd.read_csv(bdir / "shash_params.csv").set_index("gene")
    universe = [g for g in fits.index if g in shash_p.index and g in gene_names]  # model-route only

    is_hc, batch, small = data["is_hc"], data["batch"], data["small_hc_batches"]
    tr_idx = np.where(is_hc & (batch != batch_id) & ~np.isin(batch, list(small)))[0]
    scaler = StandardScaler().fit(data["X_raw"][tr_idx])

    hc_test_names = np.array(meta["test_names"])[test_is_hc]
    hc_test_rows = np.array([name2row[n] for n in hc_test_names])

    out = dict(gene_names=gene_names, gene_pos={g: j for j, g in enumerate(gene_names)},
               Z_hc=Z_test[test_is_hc], universe=universe, fits=fits, shash_p=shash_p,
               scaler=scaler, hc_test_rows=hc_test_rows)
    _cache[batch_id] = out
    return out


def mu_alpha_tau2(b, genes, sample_rows):
    Xs = b["scaler"].transform(data["X_raw"][sample_rows])
    Xa = np.column_stack([np.ones(len(sample_rows)), Xs])
    mu, alpha, tau2 = {}, {}, {}
    fits = b["fits"]
    mu_cols = [c for c in fits.columns if c.startswith("mu_coef_")]
    disp_cols = [c for c in fits.columns if c.startswith("disp_coef_")]
    for g in genes:
        row = fits.loc[g]
        mu_coef = row[mu_cols].values.astype(float)
        disp_coef = row[disp_cols].values.astype(float)
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(disp_coef, nan=0.0)) if not np.all(np.isnan(disp_coef))
                    else np.full(len(sample_rows), float(row["trend_alpha"])))
        tau2[g] = float(row["tau2"])
    return mu, alpha, tau2


def perturb_genes_fold(b, genes, sample_rows, log2fc, seed):
    # spike-in / limit-of-detection design: adds a KNOWN, deterministic amount -- the model's
    # expected mean shift mu*(2^log2fc - 1) -- on top of each sample's REAL observed count, the
    # same way an ERCC spike-in adds a known quantity of material to a real sample and asks
    # whether it's recovered against that sample's own real background noise. We are NOT claiming
    # y_pert is a valid draw from NB(mu*fold, alpha) (its variance stays at the real sample's own
    # level, not the larger variance a true NB(mu*fold, alpha) sample would have) -- so this
    # measures a detection-limit/floor, not a disease-effect-size power estimate. Chosen over
    # posterior-predictive resimulation (the field-standard polyester/splatter design) because it
    # better matches cfRNA's actual noise floor, and because filtering to well-expressed genes to
    # avoid this issue would exclude real low-expression disease-relevant transcripts that
    # legitimately spike in disease.
    mu, alpha, tau2 = mu_alpha_tau2(b, genes, sample_rows)
    shash_p = b["shash_p"]
    z_of = {}
    for g in genes:
        y_real = data["Y"][sample_rows, data["gene_col"][g]]
        y_pert = np.round(np.maximum(y_real + mu[g] * (2 ** log2fc - 1), 0))
        z_raw = marginal_nb_rqr(y_pert, mu[g], alpha[g], tau2[g], seed=seed + hash(g) % 9973)
        srow = shash_p.loc[g]
        z = shash_transform_to_z(z_raw, srow.xi, srow.eta, srow.eps, srow.delta) if srow.ok else z_raw
        z_of[g] = np.clip(z, -50, 50)
    return z_of

In [ ]:
def iter_lobo_folds():
    for batch_id in BATCHES:
        b = load_batch(batch_id)
        n_hc = len(b["hc_test_rows"])
        for boot in range(N_BOOTSTRAP):
            rng = np.random.default_rng(1000 * boot + hash(batch_id) % 9973)
            perm = rng.permutation(n_hc)
            idx_A = perm[: n_hc // 2]
            rows_A = b["hc_test_rows"][idx_A]
            yield dict(group_id=batch_id, b=b, rows_A=rows_A, idx_A=idx_A, boot=boot)


_insample_cache = None


def load_insample_model():
    global _insample_cache
    if _insample_cache is not None:
        return _insample_cache

    genes = pickle.load(open(config.ENGINE_MIXED_DIR / "genes.pkl", "rb"))
    summary = pd.read_csv(config.ENGINE_MIXED_DIR / "training_summary.csv").set_index("gene")
    scaler_ins = pickle.load(open(config.ENGINE_MIXED_DIR / "scaler.pkl", "rb"))

    universe = [g for g in summary.index[summary["ok"].fillna(False)] if g in genes and genes[g].mu_coef is not None]
    recs = [genes[g] for g in universe]
    mu_coef_mat = np.nan_to_num(np.stack([r.mu_coef for r in recs]).astype(float), nan=0.0)
    disp_coef_mat = np.stack([r.disp_coef for r in recs]).astype(float)
    has_disp = ~np.all(np.isnan(disp_coef_mat), axis=1)
    disp_coef_mat = np.nan_to_num(disp_coef_mat, nan=0.0)
    tau2_vec = np.array([r.tau2 for r in recs], dtype=float)
    trend_alpha_vec = np.array([r.trend_alpha for r in recs], dtype=float)

    su = summary.loc[universe]
    ok_arr = su["cv_shash_ok"].fillna(False).infer_objects(copy=False).values.astype(bool)
    xi_arr = np.where(ok_arr, su["cv_shash_xi"].fillna(0.0).values, 0.0)
    eta_arr = np.where(ok_arr, su["cv_shash_eta"].fillna(1.0).values, 1.0)
    eps_arr = np.where(ok_arr, su["cv_shash_eps"].fillna(0.0).values, 0.0)
    delta_arr = np.where(ok_arr, su["cv_shash_delta"].fillna(1.0).values, 1.0)

    mu_cols = [f"mu_coef_{i}" for i in range(mu_coef_mat.shape[1])]
    disp_cols = [f"disp_coef_{i}" for i in range(disp_coef_mat.shape[1])]
    fits = pd.DataFrame(mu_coef_mat, columns=mu_cols, index=universe)
    for i, c in enumerate(disp_cols):
        fits[c] = np.where(has_disp, disp_coef_mat[:, i], np.nan)
    fits["trend_alpha"] = trend_alpha_vec
    fits["tau2"] = tau2_vec
    shash_p = pd.DataFrame(dict(xi=xi_arr, eta=eta_arr, eps=eps_arr, delta=delta_arr, ok=ok_arr), index=universe)

    out = dict(gene_names=gene_names_global, gene_pos=gene_pos_global, Z_hc=Z_hc_global,
               universe=universe, fits=fits, shash_p=shash_p, scaler=scaler_ins,
               hc_test_rows=hc_global_rows)
    _insample_cache = out
    return out


def iter_insample_cv_folds():
    b = load_insample_model()
    keep = np.where(~HC_QC_OUTLIER)[0]  # drop pathologically miscalibrated HC before CV-splitting
    n = len(keep)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
    for fold, (_, idx_A_k) in enumerate(kf.split(np.arange(n))):
        idx_A = keep[idx_A_k]
        rows_A = b["hc_test_rows"][idx_A]
        yield dict(group_id=f"insample_fold{fold}", b=b, rows_A=rows_A, idx_A=idx_A, boot=fold)


SPLIT_PROVIDERS = {"lobo": iter_lobo_folds, "insample_cv": iter_insample_cv_folds}

In [ ]:
def run_per_sample(split_method):
    cache_path = PERTURB_DIR / f"sweep_persample_fold__{split_method}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for fold in SPLIT_PROVIDERS[split_method]():
        group_id, b = fold["group_id"], fold["b"]
        rows_A, idx_A, boot = fold["rows_A"], fold["idx_A"], fold["boot"]
        universe = b["universe"]

        for log2fc in LOG2FCS:
            rng = np.random.default_rng(int(1e6 * log2fc) + boot + hash(group_id) % 9973)
            pert_genes = list(rng.choice(universe, min(N_PERTURB_GENES, len(universe)), replace=False))
            z_of = perturb_genes_fold(b, pert_genes, rows_A, log2fc, seed=int(1e6 * log2fc) + boot)

            Z_A = b["Z_hc"][idx_A].copy()
            labels = np.zeros(len(b["gene_names"]), dtype=bool)
            for g in pert_genes:
                j = b["gene_pos"][g]
                Z_A[:, j] = z_of[g]
                labels[j] = True

            finite = np.isfinite(Z_A)
            p = np.where(finite, 2 * norm.sf(np.abs(np.where(finite, Z_A, 0.0))), np.nan)

            for q in Q_LEVELS:
                tp = fp = fn_ = tn = 0
                for i in range(Z_A.shape[0]):
                    fin_i = finite[i]
                    reject_i = np.zeros(Z_A.shape[1], dtype=bool)
                    reject_i[fin_i] = bh_fdr_reject(p[i, fin_i], q=q)
                    tp += int((reject_i & labels).sum())
                    fp += int((reject_i & ~labels).sum())
                    fn_ += int((~reject_i & labels).sum())
                    tn += int((~reject_i & ~labels).sum())
                rows.append(dict(batch=group_id, boot=boot, log2fc=log2fc, q=q,
                                 n_samples=int(Z_A.shape[0]), n_universe=int(Z_A.shape[1]),
                                 n_perturbed=len(pert_genes), tp=tp, fp=fp, fn=fn_, tn=tn))
        print(split_method, group_id, "per-sample done", flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df


def run_per_sample_all_splits():
    parts = []
    for split_method in ALL_SPLIT_METHODS:
        df = run_per_sample(split_method)
        df["split"] = split_method
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


sweep_persample = run_per_sample_all_splits()
sweep_persample

In [ ]:
counts_ps = sweep_persample.groupby(["split", "q", "log2fc"])[["tp", "fp", "fn", "tn"]].sum()
tp, fp, fn, tn = counts_ps["tp"], counts_ps["fp"], counts_ps["fn"], counts_ps["tn"]
summary_ps = pd.DataFrame(dict(
    TP=tp, FP=fp, FN=fn, TN=tn,
    Precision=tp / (tp + fp), Sensitivity=tp / (tp + fn), Specificity=tn / (tn + fp),
    Accuracy=(tp + tn) / (tp + fp + fn + tn), FDR=fp / (tp + fp),
    F1=2 * tp / (2 * tp + fp + fn),
)).round(3)

for sm in ALL_SPLIT_METHODS:
    print(f"per-sample (no aggregation) -- split method: {sm}")
    display(summary_ps.loc[sm])

null_calib_ps = sweep_persample[sweep_persample.log2fc == 0].groupby(["split", "q"])["fp"].agg(["mean", "max"])
print("null (log2fc=0) per-sample FP calibration:")
display(null_calib_ps)

In [ ]:
import matplotlib.pyplot as plt

_PLOT_SPLITS = ["lobo", "insample_cv"]
_METRICS = ["Sensitivity", "Precision", "FDR", "Specificity"]
_LINESTYLES = {"Sensitivity": "-", "Precision": "--", "FDR": ":", "Specificity": "-."}
_MARKERS = {"Sensitivity": "o", "Precision": "s", "FDR": "^", "Specificity": "D"}

# grayscale gradient from black -- edit this list to restyle
PALETTE = {"Sensitivity": "#000000", "Precision": "#404040", "FDR": "#808080", "Specificity": "#B0B0B0"}

_plot_df = summary_ps.reset_index()
_plot_df = _plot_df[(_plot_df.split.isin(_PLOT_SPLITS)) & (_plot_df.log2fc > 0)]
_qs = sorted(_plot_df.q.unique())

fig, axes = plt.subplots(len(_qs), len(_PLOT_SPLITS), figsize=(5 * len(_PLOT_SPLITS), 3 * len(_qs)),
                         sharex=False, sharey=True, squeeze=False)
for i, q in enumerate(_qs):
    for j, sm in enumerate(_PLOT_SPLITS):
        ax = axes[i, j]
        sub = _plot_df[(_plot_df.q == q) & (_plot_df.split == sm)].sort_values("log2fc")
        for metric in _METRICS:
            ax.plot(sub.log2fc, sub[metric], color=PALETTE[metric], linestyle=_LINESTYLES[metric],
                    marker=_MARKERS[metric], label=metric)
        ax.axhline(q, color=PALETTE["FDR"], linestyle="-", linewidth=1, alpha=0.4,
                  label="FDR target (q)" if (i, j) == (0, 0) else None)
        ax.set_ylim(-0.02, 1.02)
        if i == 0:
            ax.set_title(f"split={sm}")
        if j == 0:
            ax.set_ylabel(f"q={q}")
        if i == len(_qs) - 1:
            ax.set_xlabel("log2fc")
        ax.grid(alpha=0.2)
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
fig.suptitle("Per-sample spike-in LOD: detection metrics vs log2fc, by FDR level q and split method", y=1.02)
plt.tight_layout()
plt.show()